# HAR regression with regularisation (ridge)

The three richest **log** HAR specifications from `SimpleHAR_Regressions.ipynb` all add
GVZ (and optionally another asset's RV), so their regressors — gold daily/weekly/monthly
log-RV, log-GVZ, log-SPX / log-crude — are highly collinear, which makes the OLS
coefficients unstable. This notebook re-estimates those three specs with **ridge
regression** to shrink the collinear coefficients, choosing the ridge penalty (alpha) by a
**simple in-sample grid search re-run inside each rolling window** (leave-one-out / GCV,
no look-ahead), and compares the ridge out-of-sample QLIKE against the OLS baseline.

Ridge runs (analogues of the OLS log+GVZ runs in the source notebook):
- **Run 15** — ridge of Run 4:  log HAR + log GVZ
- **Run 16** — ridge of Run 10: log HAR + log SPX + log GVZ
- **Run 17** — ridge of Run 14: log HAR + log crude + log GVZ

In [1]:
# ===========================================================================
# Cell 1 — Imports & data
# ===========================================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

# Same aligned daily realized-variance panel used by SimpleHAR_Regressions.ipynb
data = pd.read_parquet("merged_RV_GVZ.parquet")
rv = data["RV_gold"].astype(float)

WINDOW = 500      # rolling estimation window (matches the OLS runs)
EPS = 1e-6        # QLIKE forecast floor

print(f"RV_gold: {len(rv)} obs, {rv.index.min().date()} .. {rv.index.max().date()}")
print(f"Columns available: {list(data.columns)}")

RV_gold: 4114 obs, 2010-01-04 .. 2026-05-29
Columns available: ['RV_gold', 'RV_crude', 'RV_ES', 'GVZ_close']


In [2]:
# ===========================================================================
# Cell 2 — Build the 3 log+GVZ design tables (mirror SimpleHAR Cells 7 & 11)
# ===========================================================================
# Log-HAR components on x = log(RV_gold):
#   x_d[t]=x_t, x_w[t]=mean(x_{t-4..t}), x_m[t]=mean(x_{t-21..t})
#   y_log[t]=x_{t+1}=log(RV_{t+1}); y_level[t]=RV_{t+1} (kept for QLIKE in levels).
# Exogenous day-t terms are logged and known at the close (no look-ahead).

# Strict positivity required for every logged input
for col in ["RV_gold", "GVZ_close", "RV_ES", "RV_crude"]:
    assert (data[col] > 0).all(), f"{col} has non-positive values; log undefined"

x = np.log(rv)

def build_log_design(extra_cols):
    """Base log-HAR design + optional logged exogenous day-t columns (dict name->series)."""
    df = pd.DataFrame(index=rv.index)
    df["x_d"] = x
    df["x_w"] = x.rolling(5).mean()
    df["x_m"] = x.rolling(22).mean()
    for name, series in extra_cols.items():
        df[name] = series.reindex(rv.index)
    df["y_log"]   = x.shift(-1)        # log(RV_{t+1})
    df["y_level"] = rv.shift(-1)       # RV_{t+1} in levels, for QLIKE
    return df.dropna()

log_gvz   = np.log(data["GVZ_close"])
log_spx   = np.log(data["RV_ES"])
log_crude = np.log(data["RV_crude"])

# Run 15 / 16 / 17 design tables
d_gvz       = build_log_design({"log_GVZ": log_gvz})
d_spx_gvz   = build_log_design({"log_GVZ": log_gvz, "log_RV_ES": log_spx})
d_crude_gvz = build_log_design({"log_GVZ": log_gvz, "log_RV_crude": log_crude})

for name, df in [("d_gvz", d_gvz), ("d_spx_gvz", d_spx_gvz), ("d_crude_gvz", d_crude_gvz)]:
    assert df.notna().all().all(), f"unexpected NaNs in {name}"

print("Run 15  log+GVZ        cols:", list(d_gvz.columns),       f"({len(d_gvz)} rows)")
print("Run 16  log+SPX+GVZ    cols:", list(d_spx_gvz.columns),   f"({len(d_spx_gvz)} rows)")
print("Run 17  log+crude+GVZ  cols:", list(d_crude_gvz.columns), f"({len(d_crude_gvz)} rows)")
d_gvz.head()

Run 15  log+GVZ        cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'y_log', 'y_level'] (4092 rows)
Run 16  log+SPX+GVZ    cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'log_RV_ES', 'y_log', 'y_level'] (4092 rows)
Run 17  log+crude+GVZ  cols: ['x_d', 'x_w', 'x_m', 'log_GVZ', 'log_RV_crude', 'y_log', 'y_level'] (4092 rows)


,x_d,x_w,x_m,log_GVZ,y_log,y_level
Date,,,,,,
2010-02-03,2.850663,2.981362,2.933665,3.095125,3.580491,35.891145
2010-02-04,3.580491,3.080406,2.967351,3.302849,3.444815,31.337489
2010-02-05,3.444815,3.158204,2.994161,3.317453,2.911737,18.388714
2010-02-08,2.911737,3.146146,2.996290,3.291383,2.940123,18.918172
2010-02-09,2.940123,3.145566,3.005753,3.236716,3.123939,22.735769


In [3]:
# ===========================================================================
# Cell 3 — Shared helpers: QLIKE, OLS baseline, ridge with per-window alpha
# ===========================================================================
# QLIKE (Patton 2011 robust form), floored so log is defined.
def _qlike(actual, forecast, eps=EPS):
    f = np.maximum(forecast, eps)
    r = actual / f
    return r - np.log(r) - 1.0, int((forecast <= eps).sum())

# --- OLS baseline (identical protocol to SimpleHAR Cell 13) ---
# 500-day rolling log-space OLS + Duan smearing back-transform -> avg QLIKE.
def rolling_log_qlike_smearing(design, feat_cols, ylog_col, ylevel_col, window=WINDOW):
    X = np.column_stack([np.ones(len(design)), design[feat_cols].to_numpy()])
    yl  = design[ylog_col].to_numpy()
    lvl = design[ylevel_col].to_numpy()
    fc, ac = [], []
    for t in range(window, len(design)):
        Xw, yw = X[t - window:t], yl[t - window:t]
        beta, *_ = np.linalg.lstsq(Xw, yw, rcond=None)
        smearing = np.mean(np.exp(yw - Xw @ beta))          # Duan factor
        fc.append(np.exp(X[t] @ beta) * smearing); ac.append(lvl[t])
    q, clip = _qlike(np.array(ac), np.array(fc))
    return q.mean(), len(q), clip

# --- Ridge: same rolling protocol, but per-window standardised RidgeCV ---
# RidgeCV(cv=None) uses efficient leave-one-out / GCV over the alpha grid using ONLY
# the in-window training data -> the "simple in-sample grid search" for alpha (no
# look-ahead). Standardisation matters because the ridge penalty is scale-dependent.
ALPHAS = np.logspace(-4, 4, 50)

def rolling_log_ridge_qlike(design, feat_cols, ylog_col, ylevel_col,
                            alphas=ALPHAS, window=WINDOW):
    Xf  = design[feat_cols].to_numpy()
    yl  = design[ylog_col].to_numpy()
    lvl = design[ylevel_col].to_numpy()
    fc, ac, alpha_sel = [], [], []
    for t in range(window, len(design)):
        Xw, yw = Xf[t - window:t], yl[t - window:t]
        scaler = StandardScaler().fit(Xw)
        Xw_s = scaler.transform(Xw)
        model = RidgeCV(alphas=alphas, fit_intercept=True).fit(Xw_s, yw)  # LOO/GCV
        x_next = scaler.transform(Xf[t:t + 1])
        x_hat = model.predict(x_next)[0]
        smearing = np.mean(np.exp(yw - model.predict(Xw_s)))              # Duan factor
        fc.append(np.exp(x_hat) * smearing); ac.append(lvl[t])
        alpha_sel.append(model.alpha_)
    q, clip = _qlike(np.array(ac), np.array(fc))
    return q.mean(), len(q), clip, np.array(alpha_sel)

print(f"Helpers ready. Ridge alpha grid: {len(ALPHAS)} pts in [{ALPHAS.min():.0e}, {ALPHAS.max():.0e}]")

Helpers ready. Ridge alpha grid: 50 pts in [1e-04, 1e+04]


In [4]:
# ===========================================================================
# Cell 4 — Run OLS baselines + ridge (runs 15-17)
# ===========================================================================
specs = [
    ("Run 15  log+GVZ",       d_gvz,       ["x_d", "x_w", "x_m", "log_GVZ"]),
    ("Run 16  log+SPX+GVZ",   d_spx_gvz,   ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_ES"]),
    ("Run 17  log+crude+GVZ", d_crude_gvz, ["x_d", "x_w", "x_m", "log_GVZ", "log_RV_crude"]),
]

rows = []
for label, design, feats in specs:
    q_ols, n_ols, _ = rolling_log_qlike_smearing(design, feats, "y_log", "y_level")
    q_rdg, n_rdg, clip_rdg, alphas = rolling_log_ridge_qlike(design, feats, "y_log", "y_level")
    rows.append({"spec": label, "OLS_QLIKE": q_ols, "Ridge_QLIKE": q_rdg,
                 "delta": q_rdg - q_ols, "mean_alpha": alphas.mean(),
                 "median_alpha": np.median(alphas), "n_forecasts": n_rdg})
    print(f"{label:<24} OLS {q_ols:.6f} | Ridge {q_rdg:.6f}  "
          f"(Δ {q_rdg - q_ols:+.6f})  alpha mean={alphas.mean():.4g}, "
          f"median={np.median(alphas):.4g}")

Run 15  log+GVZ          OLS 0.030577 | Ridge 0.030637  (Δ +0.000060)  alpha mean=12.13, median=11.51


Run 16  log+SPX+GVZ      OLS 0.030723 | Ridge 0.030784  (Δ +0.000062)  alpha mean=13.51, median=11.51


Run 17  log+crude+GVZ    OLS 0.030618 | Ridge 0.030679  (Δ +0.000061)  alpha mean=13.49, median=11.51


In [5]:
# ===========================================================================
# Cell 5 — Consolidated OLS-vs-ridge comparison
# ===========================================================================
pd.set_option("display.float_format", lambda v: f"{v:.6f}")
comparison = pd.DataFrame(rows).set_index("spec")
print("Ridge vs OLS (lower QLIKE = better 1-day-ahead forecast):")
print(comparison.to_string())

improved = comparison.index[comparison["delta"] < 0].tolist()
print(f"\nRidge improved QLIKE on: {improved if improved else 'none'}")

best_ols   = comparison["OLS_QLIKE"].idxmin()
best_ridge = comparison["Ridge_QLIKE"].idxmin()
best_all   = comparison[["OLS_QLIKE", "Ridge_QLIKE"]].min().idxmin()
best_val   = comparison[["OLS_QLIKE", "Ridge_QLIKE"]].min().min()
print(f"Best OLS spec:   {best_ols} ({comparison.loc[best_ols, 'OLS_QLIKE']:.6f})")
print(f"Best ridge spec: {best_ridge} ({comparison.loc[best_ridge, 'Ridge_QLIKE']:.6f})")
print(f"Overall best:    {best_all} via {best_val:.6f}")

Ridge vs OLS (lower QLIKE = better 1-day-ahead forecast):
                       OLS_QLIKE  Ridge_QLIKE    delta  mean_alpha  median_alpha  n_forecasts
spec                                                                                         
Run 15  log+GVZ         0.030577     0.030637 0.000060   12.133401     11.513954         3592
Run 16  log+SPX+GVZ     0.030723     0.030784 0.000062   13.507552     11.513954         3592
Run 17  log+crude+GVZ   0.030618     0.030679 0.000061   13.490154     11.513954         3592

Ridge improved QLIKE on: none
Best OLS spec:   Run 15  log+GVZ (0.030577)
Best ridge spec: Run 15  log+GVZ (0.030637)
Overall best:    OLS_QLIKE via 0.030577
